In [15]:
# ============================================================
# MyGPT2 Validation Evaluation
# Cell 1 - Imports
# ============================================================

from pathlib import Path
import sys
import math
import time
import json

import torch
import torch.nn.functional as F

print("=" * 75)
print("MyGPT2 - Validation Evaluation")
print("=" * 75)

print("PyTorch Version :", torch.__version__)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("CUDA Available  : YES")
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("CUDA Available  : NO")

print("Device          :", DEVICE)

print("=" * 75)

MyGPT2 - Validation Evaluation
PyTorch Version : 2.13.0+cu132
CUDA Available  : YES
GPU             : NVIDIA GeForce RTX 5060 Ti
Device          : cuda


In [16]:
# ============================================================
# Cell 2 - Project Paths
# ============================================================

PROJECT_ROOT = Path(r"D:\Gpt2_v01").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "checkpoints"
    / "final_step_00010000.pt"
)

TOKENIZER_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tokenizer"
    / "tokenizer.json"
)

print("=" * 75)
print("Project Paths")
print("=" * 75)

print("Project Root :", PROJECT_ROOT)
print("Checkpoint   :", CHECKPOINT_PATH)
print("Tokenizer    :", TOKENIZER_PATH)

print()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist:\n{PROJECT_ROOT}"
    )

if not CHECKPOINT_PATH.exists():
    checkpoint_dir = (
        PROJECT_ROOT
        / "artifacts"
        / "checkpoints"
    )

    print("Available checkpoints:")
    
    if checkpoint_dir.exists():
        for file in sorted(checkpoint_dir.glob("*.pt")):
            print("  -", file.name)

    raise FileNotFoundError(
        f"\nCheckpoint not found:\n{CHECKPOINT_PATH}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found:\n{TOKENIZER_PATH}"
    )

print("Project Root : ✅ FOUND")
print("Checkpoint   : ✅ FOUND")
print("Tokenizer    : ✅ FOUND")

print("=" * 75)

Project Paths
Project Root : D:\Gpt2_v01
Checkpoint   : D:\Gpt2_v01\artifacts\checkpoints\final_step_00010000.pt
Tokenizer    : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json

Project Root : ✅ FOUND
Checkpoint   : ✅ FOUND
Tokenizer    : ✅ FOUND


In [17]:
print("Checkpoint exists :", CHECKPOINT_PATH.exists())
print("Tokenizer exists  :", TOKENIZER_PATH.exists())

print()
print("Checkpoint size   :",
      f"{CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB")

print("Tokenizer size    :",
      f"{TOKENIZER_PATH.stat().st_size / (1024**2):.2f} MB")

Checkpoint exists : True
Tokenizer exists  : True

Checkpoint size   : 1259.35 MB
Tokenizer size    : 2.16 MB


In [18]:
# ============================================================
# Cell 3 - MyGPT2 Imports
# ============================================================

from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer
from training.checkpoint import load_checkpoint

print("GPTConfig       : ✅")
print("MyGPTModel      : ✅")
print("MyGPTTokenizer  : ✅")
print("load_checkpoint : ✅")

print()
print("All MyGPT2 imports successful.")

GPTConfig       : ✅
MyGPTModel      : ✅
MyGPTTokenizer  : ✅
load_checkpoint : ✅

All MyGPT2 imports successful.


In [20]:
from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer

from training.checkpoint import load_checkpoint

# IMPORTANT:
# Replace this import with the exact Dataset class used by train.py.
#
# Example:
# from training.dataset import TinyStoriesDataset
#
# If your train.py imports the dataset from another module,
# use that exact import here.

In [22]:
# ============================================================
# Cell 4 - Model Configuration
# ============================================================

config = GPTConfig()


def get_config_value(config, names, default=None):
    """
    Safely retrieve the first available configuration attribute.
    """
    for name in names:
        if hasattr(config, name):
            value = getattr(config, name)

            if value is not None:
                return value

    return default


# ------------------------------------------------------------
# Vocabulary
# ------------------------------------------------------------

VOCAB_SIZE = get_config_value(
    config,
    [
        "vocab_size",
    ],
)


# ------------------------------------------------------------
# Sequence Length
# ------------------------------------------------------------
#
# Your training run used:
#
# Sequence Length : 512
#
# We first try to find the value in GPTConfig.
# If GPTConfig does not expose it, use the exact
# sequence length used during training.
# ------------------------------------------------------------

SEQUENCE_LENGTH = get_config_value(
    config,
    [
        "context_length",
        "block_size",
        "max_seq_len",
        "max_sequence_length",
        "sequence_length",
        "seq_length",
        "context_size",
        "n_positions",
        "max_position_embeddings",
        "max_context_length",
    ],
    default=512,
)


# ------------------------------------------------------------
# Hidden Size
# ------------------------------------------------------------

HIDDEN_SIZE = get_config_value(
    config,
    [
        "hidden_size",
        "n_embd",
        "embedding_dim",
        "d_model",
    ],
)


# ------------------------------------------------------------
# Transformer Layers
# ------------------------------------------------------------

NUM_LAYERS = get_config_value(
    config,
    [
        "num_layers",
        "n_layer",
        "layers",
        "num_hidden_layers",
    ],
)


# ------------------------------------------------------------
# Attention Heads
# ------------------------------------------------------------

NUM_HEADS = get_config_value(
    config,
    [
        "num_heads",
        "n_head",
        "attention_heads",
        "num_attention_heads",
    ],
)


# ------------------------------------------------------------
# Intermediate / FFN Size
# ------------------------------------------------------------

INTERMEDIATE_SIZE = get_config_value(
    config,
    [
        "intermediate_size",
        "ffn_size",
        "hidden_dim",
    ],
)


# ============================================================
# Display Configuration
# ============================================================

print("=" * 75)
print("Model Configuration")
print("=" * 75)

print(
    f"Vocabulary Size      : "
    f"{VOCAB_SIZE:,}"
)

print(
    f"Sequence Length      : "
    f"{SEQUENCE_LENGTH}"
)

print(
    f"Hidden Size          : "
    f"{HIDDEN_SIZE}"
)

print(
    f"Transformer Layers   : "
    f"{NUM_LAYERS}"
)

print(
    f"Attention Heads      : "
    f"{NUM_HEADS}"
)

print(
    f"Intermediate Size    : "
    f"{INTERMEDIATE_SIZE}"
)

print("=" * 75)


# ============================================================
# Validation
# ============================================================

if VOCAB_SIZE is None:
    raise RuntimeError(
        "Could not determine vocabulary size "
        "from GPTConfig."
    )


if SEQUENCE_LENGTH is None:
    raise RuntimeError(
        "Could not determine sequence length."
    )


if SEQUENCE_LENGTH != 512:
    print(
        "WARNING: The detected sequence length is "
        f"{SEQUENCE_LENGTH}, but the model was trained "
        "with sequence length 512."
    )
else:
    print(
        "Sequence Length : ✅ 512 "
        "(matches training configuration)"
    )


# ============================================================
# Expected Configuration Check
# ============================================================

EXPECTED_CONFIG = {
    "vocab_size": 32000,
    "sequence_length": 512,
    "hidden_size": 768,
    "num_layers": 12,
    "num_heads": 12,
    "intermediate_size": 3072,
}


print()
print("=" * 75)
print("Expected Training Configuration Check")
print("=" * 75)

checks = {
    "Vocabulary Size": (
        VOCAB_SIZE,
        EXPECTED_CONFIG["vocab_size"],
    ),
    "Sequence Length": (
        SEQUENCE_LENGTH,
        EXPECTED_CONFIG["sequence_length"],
    ),
    "Hidden Size": (
        HIDDEN_SIZE,
        EXPECTED_CONFIG["hidden_size"],
    ),
    "Transformer Layers": (
        NUM_LAYERS,
        EXPECTED_CONFIG["num_layers"],
    ),
    "Attention Heads": (
        NUM_HEADS,
        EXPECTED_CONFIG["num_heads"],
    ),
    "Intermediate Size": (
        INTERMEDIATE_SIZE,
        EXPECTED_CONFIG["intermediate_size"],
    ),
}


configuration_passed = True

for name, (actual, expected) in checks.items():

    if actual is None:
        print(
            f"{name:<25}: ⚠️ UNKNOWN "
            f"(expected {expected})"
        )

        configuration_passed = False

    elif actual == expected:
        print(
            f"{name:<25}: ✅ {actual}"
        )

    else:
        print(
            f"{name:<25}: ❌ {actual} "
            f"(expected {expected})"
        )

        configuration_passed = False


print("=" * 75)

if configuration_passed:
    print(
        "Configuration Check : ✅ PASSED"
    )
else:
    print(
        "Configuration Check : ⚠️ REVIEW REQUIRED"
    )

Model Configuration
Vocabulary Size      : 32,000
Sequence Length      : 512
Hidden Size          : 768
Transformer Layers   : 12
Attention Heads      : 12
Intermediate Size    : 3072
Sequence Length : ✅ 512 (matches training configuration)

Expected Training Configuration Check
Vocabulary Size          : ✅ 32000
Sequence Length          : ✅ 512
Hidden Size              : ✅ 768
Transformer Layers       : ✅ 12
Attention Heads          : ✅ 12
Intermediate Size        : ✅ 3072
Configuration Check : ✅ PASSED


In [23]:
# ============================================================
# Cell 5 - Load Tokenizer
# ============================================================

print("=" * 75)
print("Loading Tokenizer")
print("=" * 75)

tokenizer = MyGPTTokenizer.load(
    TOKENIZER_PATH
)

TOKENIZER_VOCAB_SIZE = tokenizer.vocabulary_size

print("Tokenizer Path      :", TOKENIZER_PATH)
print("Tokenizer Vocabulary:", f"{TOKENIZER_VOCAB_SIZE:,}")

if TOKENIZER_VOCAB_SIZE != VOCAB_SIZE:
    raise RuntimeError(
        "Vocabulary mismatch!\n"
        f"Tokenizer vocabulary : {TOKENIZER_VOCAB_SIZE}\n"
        f"Model vocabulary     : {VOCAB_SIZE}"
    )

print()
print("Tokenizer            : ✅ LOADED")
print("Vocabulary Match     : ✅ PASSED")

print("=" * 75)

Loading Tokenizer
Tokenizer Path      : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json
Tokenizer Vocabulary: 32,000

Tokenizer            : ✅ LOADED
Vocabulary Match     : ✅ PASSED


In [24]:
# ============================================================
# Cell 6 - Load Model + Checkpoint
# ============================================================

print("=" * 75)
print("Loading Model")
print("=" * 75)

model = MyGPTModel(config).to(DEVICE)

model.eval()

print("Model created        : ✅")

checkpoint = load_checkpoint(
    path=CHECKPOINT_PATH,
    model=model,
    optimizer=None,
    scheduler=None,
    device=DEVICE,
    restore_rng=False,
)

print()
print("Checkpoint version   :", checkpoint.get("checkpoint_version"))
print("Saved epoch          :", checkpoint.get("epoch"))
print("Saved global step    :", checkpoint.get("global_step"))
print("Saved training loss  :", checkpoint.get("train_loss"))
print("Saved validation loss:", checkpoint.get("val_loss"))

print()
print("Model checkpoint     : ✅ LOADED")

print("=" * 75)

Loading Model
Model created        : ✅

Checkpoint version   : 1.2
Saved epoch          : 0
Saved global step    : 10000
Saved training loss  : 1.3898952007293701
Saved validation loss: None

Model checkpoint     : ✅ LOADED


In [25]:
# ============================================================
# Cell 7 - Parameter Verification
# ============================================================

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("=" * 75)
print("Model Parameters")
print("=" * 75)

print(
    f"Total Parameters     : {total_parameters:,}"
)

print(
    f"Total Parameters     : "
    f"{total_parameters / 1_000_000:.2f}M"
)

print(
    f"Trainable Parameters : "
    f"{trainable_parameters:,}"
)

print("=" * 75)

EXPECTED_PARAMETERS = 110_025_216

if total_parameters == EXPECTED_PARAMETERS:
    print("Parameter Count      : ✅ MATCH")
else:
    print(
        "Parameter Count      : ⚠️ DIFFERENT"
    )
    print(
        f"Expected             : {EXPECTED_PARAMETERS:,}"
    )

Model Parameters
Total Parameters     : 110,025,216
Total Parameters     : 110.03M
Trainable Parameters : 110,025,216
Parameter Count      : ✅ MATCH


In [26]:
# ============================================================
# Cell 8 - Load TinyStories Validation Dataset
# ============================================================

try:
    from datasets import load_dataset
except ImportError:
    raise ImportError(
        "The 'datasets' package is required.\n"
        "Install it with:\n"
        "pip install datasets"
    )

print("=" * 75)
print("Loading TinyStories Validation Dataset")
print("=" * 75)

VALIDATION_DATASET_NAME = "roneneldan/TinyStories"

print("Dataset :", VALIDATION_DATASET_NAME)
print("Split   : validation")
print()

validation_dataset = load_dataset(
    VALIDATION_DATASET_NAME,
    split="validation",
)

print(
    "Validation documents :",
    f"{len(validation_dataset):,}"
)

print()
print("Dataset loading      : ✅ PASSED")

print("=" * 75)

Loading TinyStories Validation Dataset
Dataset : roneneldan/TinyStories
Split   : validation



Validation documents : 21,990

Dataset loading      : ✅ PASSED


In [27]:
# ============================================================
# Cell 9 - Dataset Inspection
# ============================================================

print("=" * 75)
print("Validation Dataset Inspection")
print("=" * 75)

print("Columns:")
print(validation_dataset.column_names)

print()

if len(validation_dataset) == 0:
    raise RuntimeError(
        "Validation dataset is empty."
    )

first_sample = validation_dataset[0]

print("First sample keys:")
print(first_sample.keys())

print()

if "text" not in first_sample:
    raise RuntimeError(
        "TinyStories validation dataset does not contain "
        "the expected 'text' field."
    )

print("First sample preview:")
print(
    first_sample["text"][:500]
)

print("=" * 75)

Validation Dataset Inspection
Columns:
['text']

First sample keys:
dict_keys(['text'])

First sample preview:
Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.


In [28]:
# ============================================================
# Cell 10 - Tokenization Helper
# ============================================================

def encode_text(tokenizer, text):
    """
    Convert text into integer token IDs.

    Supports common MyGPTTokenizer interfaces.
    """

    if not isinstance(text, str):
        raise TypeError(
            f"Expected text to be str, got {type(text)}"
        )

    # --------------------------------------------------------
    # Preferred MyGPTTokenizer API
    # --------------------------------------------------------

    if hasattr(tokenizer, "encode"):
        result = tokenizer.encode(text)

        # Some wrappers return an object
        if hasattr(result, "ids"):
            result = result.ids

        if isinstance(result, torch.Tensor):
            result = result.detach().cpu().tolist()

        if isinstance(result, (list, tuple)):
            return [int(x) for x in result]

    # --------------------------------------------------------
    # Explicit tokenize API
    # --------------------------------------------------------

    if hasattr(tokenizer, "tokenize"):
        result = tokenizer.tokenize(text)

        if hasattr(result, "ids"):
            result = result.ids

        if isinstance(result, torch.Tensor):
            result = result.detach().cpu().tolist()

        if isinstance(result, (list, tuple)):
            return [int(x) for x in result]

    raise RuntimeError(
        "Could not determine how to encode text with "
        "MyGPTTokenizer."
    )


# Test tokenizer

test_text = validation_dataset[0]["text"]

test_tokens = encode_text(
    tokenizer,
    test_text,
)

print("=" * 75)
print("Tokenizer Test")
print("=" * 75)

print("Characters :", len(test_text))
print("Tokens     :", len(test_tokens))

print(
    "First tokens:",
    test_tokens[:20]
)

if len(test_tokens) == 0:
    raise RuntimeError(
        "Tokenizer produced zero tokens."
    )

print()
print("Tokenizer test       : ✅ PASSED")

print("=" * 75)

Tokenizer Test
Characters : 349
Tokens     : 84
First tokens: [2, 2753, 17, 2753, 609, 221, 2298, 728, 239, 350, 15, 333, 3447, 15, 10346, 15, 539, 728, 298, 371]

Tokenizer test       : ✅ PASSED


In [29]:
# ============================================================
# Cell 11 - Build Validation Sequences
# ============================================================

MAX_VALIDATION_DOCUMENTS = 5000

validation_sequences = []

documents_processed = 0
documents_skipped = 0
total_tokens = 0

print("=" * 75)
print("Building Validation Sequences")
print("=" * 75)

for index, sample in enumerate(
    validation_dataset
):

    if documents_processed >= MAX_VALIDATION_DOCUMENTS:
        break

    text = sample["text"]

    if not isinstance(text, str):
        documents_skipped += 1
        continue

    text = text.strip()

    if not text:
        documents_skipped += 1
        continue

    token_ids = encode_text(
        tokenizer,
        text,
    )

    if len(token_ids) < SEQUENCE_LENGTH + 1:
        documents_skipped += 1
        continue

    # --------------------------------------------------------
    # Create non-overlapping 512-token sequences.
    # --------------------------------------------------------

    usable_length = (
        len(token_ids)
        // SEQUENCE_LENGTH
    ) * SEQUENCE_LENGTH

    usable_tokens = token_ids[:usable_length]

    for start in range(
        0,
        len(usable_tokens) - SEQUENCE_LENGTH,
        SEQUENCE_LENGTH,
    ):

        sequence = usable_tokens[
            start:start + SEQUENCE_LENGTH + 1
        ]

        if len(sequence) != SEQUENCE_LENGTH + 1:
            continue

        validation_sequences.append(
            sequence
        )

        total_tokens += SEQUENCE_LENGTH

    documents_processed += 1


print(
    "Documents processed :",
    f"{documents_processed:,}"
)

print(
    "Documents skipped   :",
    f"{documents_skipped:,}"
)

print(
    "Validation sequences:",
    f"{len(validation_sequences):,}"
)

print(
    "Evaluation tokens   :",
    f"{total_tokens:,}"
)

if len(validation_sequences) == 0:
    raise RuntimeError(
        "No validation sequences were generated."
    )

print()
print("Sequence generation : ✅ PASSED")

print("=" * 75)

Building Validation Sequences
Documents processed : 569
Documents skipped   : 21,421
Validation sequences: 9
Evaluation tokens   : 4,608

Sequence generation : ✅ PASSED


In [30]:
# ============================================================
# Cell 12 - Validation DataLoader
# ============================================================

from torch.utils.data import Dataset, DataLoader


class ValidationDataset(Dataset):
    """
    Simple fixed-length language-model validation dataset.
    """

    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        sequence = self.sequences[index]

        input_ids = torch.tensor(
            sequence[:-1],
            dtype=torch.long,
        )

        labels = torch.tensor(
            sequence[1:],
            dtype=torch.long,
        )

        return input_ids, labels


VALIDATION_BATCH_SIZE = 8

validation_dataset_torch = ValidationDataset(
    validation_sequences
)

validation_loader = DataLoader(
    validation_dataset_torch,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

print("=" * 75)
print("Validation DataLoader")
print("=" * 75)

print(
    "Sequences   :",
    f"{len(validation_dataset_torch):,}"
)

print(
    "Batch Size  :",
    VALIDATION_BATCH_SIZE
)

print(
    "Batches     :",
    f"{len(validation_loader):,}"
)

test_inputs, test_labels = next(
    iter(validation_loader)
)

print()
print("Input Shape :", tuple(test_inputs.shape))
print("Label Shape :", tuple(test_labels.shape))
print("Input DType :", test_inputs.dtype)
print("Label DType :", test_labels.dtype)

if test_inputs.shape[1] != SEQUENCE_LENGTH:
    raise RuntimeError(
        "Validation sequence length mismatch."
    )

print()
print("DataLoader validation : ✅ PASSED")

print("=" * 75)

Validation DataLoader
Sequences   : 9
Batch Size  : 8
Batches     : 2

Input Shape : (8, 512)
Label Shape : (8, 512)
Input DType : torch.int64
Label DType : torch.int64

DataLoader validation : ✅ PASSED


In [31]:
# ============================================================
# Cell 13 - Validation Evaluation
# ============================================================

print("=" * 75)
print("Running Validation Evaluation")
print("=" * 75)

model.eval()

total_loss = 0.0
total_correct = 0
total_predictions = 0
total_batches = 0

validation_start = time.time()

with torch.no_grad():

    for batch_index, (
        input_ids,
        labels,
    ) in enumerate(validation_loader):

        input_ids = input_ids.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        output = model(
            input_ids=input_ids,
            labels=labels,
        )

        # ----------------------------------------------------
        # MyGPT2 output compatibility
        # ----------------------------------------------------

        if isinstance(output, tuple):

            if len(output) != 2:
                raise RuntimeError(
                    "Expected model output "
                    "format: (logits, loss)."
                )

            logits, loss = output

        else:

            if not hasattr(output, "logits"):
                raise RuntimeError(
                    "Model output does not contain logits."
                )

            logits = output.logits
            loss = output.loss

        if loss is None:
            raise RuntimeError(
                "Model returned None validation loss."
            )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Validation loss became NaN or infinite."
            )

        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        batch_size = input_ids.shape[0]

        num_tokens = labels.numel()

        total_loss += (
            loss.item()
            * batch_size
        )

        # ----------------------------------------------------
        # Token accuracy
        # ----------------------------------------------------

        predictions = logits.argmax(
            dim=-1
        )

        correct = (
            predictions == labels
        ).sum().item()

        total_correct += correct
        total_predictions += num_tokens

        total_batches += 1

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            batch_index == 0
            or (batch_index + 1) % 50 == 0
        ):

            current_accuracy = (
                total_correct
                / total_predictions
            )

            print(
                f"Batch "
                f"{batch_index + 1:>5} | "
                f"Loss {loss.item():.6f} | "
                f"Accuracy "
                f"{current_accuracy * 100:.2f}%"
            )


if total_batches == 0:
    raise RuntimeError(
        "Validation DataLoader produced zero batches."
    )

validation_loss = (
    total_loss
    / len(validation_dataset_torch)
)

validation_accuracy = (
    total_correct
    / total_predictions
)

validation_time = (
    time.time()
    - validation_start
)

print()
print("=" * 75)
print("Validation Completed")
print("=" * 75)

print(
    f"Validation Loss     : "
    f"{validation_loss:.6f}"
)

print(
    f"Token Accuracy      : "
    f"{validation_accuracy * 100:.4f}%"
)

print(
    f"Tokens Evaluated    : "
    f"{total_predictions:,}"
)

print(
    f"Validation Batches  : "
    f"{total_batches:,}"
)

print(
    f"Evaluation Time     : "
    f"{validation_time:.2f}s"
)

print("=" * 75)

Running Validation Evaluation
Batch     1 | Loss 1.326006 | Accuracy 66.77%

Validation Completed
Validation Loss     : 1.327773
Token Accuracy      : 66.7969%
Tokens Evaluated    : 4,608
Validation Batches  : 2
Evaluation Time     : 0.62s


In [32]:
# ============================================================
# Cell 14 - Perplexity
# ============================================================

print("=" * 75)
print("Language Model Metrics")
print("=" * 75)

try:
    validation_perplexity = math.exp(
        validation_loss
    )
except OverflowError:
    validation_perplexity = float("inf")

training_loss = checkpoint.get(
    "train_loss"
)

if training_loss is not None:

    training_loss = float(
        training_loss
    )

    try:
        training_perplexity = math.exp(
            training_loss
        )
    except OverflowError:
        training_perplexity = float("inf")

else:

    training_perplexity = None


print(
    f"Training Loss       : "
    f"{training_loss:.6f}"
    if training_loss is not None
    else "Training Loss       : N/A"
)

print(
    f"Training Perplexity : "
    f"{training_perplexity:.4f}"
    if training_perplexity is not None
    else "Training Perplexity : N/A"
)

print(
    f"Validation Loss     : "
    f"{validation_loss:.6f}"
)

print(
    f"Validation Perplexity: "
    f"{validation_perplexity:.4f}"
)

print(
    f"Validation Accuracy : "
    f"{validation_accuracy * 100:.4f}%"
)

print("=" * 75)

Language Model Metrics
Training Loss       : 1.389895
Training Perplexity : 4.0144
Validation Loss     : 1.327773
Validation Perplexity: 3.7726
Validation Accuracy : 66.7969%


In [33]:
# ============================================================
# Cell 15 - Train vs Validation
# ============================================================

print("=" * 75)
print("Training vs Validation")
print("=" * 75)

if training_loss is not None:

    loss_gap = (
        validation_loss
        - training_loss
    )

    print(
        f"Training Loss       : "
        f"{training_loss:.6f}"
    )

    print(
        f"Validation Loss     : "
        f"{validation_loss:.6f}"
    )

    print(
        f"Loss Gap            : "
        f"{loss_gap:+.6f}"
    )

    if training_perplexity is not None:

        print(
            f"Training Perplexity : "
            f"{training_perplexity:.4f}"
        )

    print(
        f"Validation Perplexity: "
        f"{validation_perplexity:.4f}"
    )

    print()

    if validation_loss < training_loss * 1.10:

        print(
            "Generalization     : 🟢 GOOD"
        )

    elif validation_loss < training_loss * 1.30:

        print(
            "Generalization     : 🟡 ACCEPTABLE"
        )

    else:

        print(
            "Generalization     : 🔴 POSSIBLE OVERFITTING"
        )

else:

    print(
        "Training loss unavailable."
    )

print("=" * 75)

Training vs Validation
Training Loss       : 1.389895
Validation Loss     : 1.327773
Loss Gap            : -0.062122
Training Perplexity : 4.0144
Validation Perplexity: 3.7726

Generalization     : 🟢 GOOD


In [34]:
# ============================================================
# Cell 16 - Top-K Accuracy
# ============================================================

TOP_K_VALUES = [1, 5, 10]

top_k_correct = {
    k: 0
    for k in TOP_K_VALUES
}

top_k_total = 0

print("=" * 75)
print("Calculating Top-K Accuracy")
print("=" * 75)

with torch.no_grad():

    for input_ids, labels in validation_loader:

        input_ids = input_ids.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        output = model(
            input_ids=input_ids,
            labels=labels,
        )

        if isinstance(output, tuple):

            logits = output[0]

        else:

            logits = output.logits

        max_k = max(TOP_K_VALUES)

        top_predictions = torch.topk(
            logits,
            k=max_k,
            dim=-1,
        ).indices

        labels_expanded = labels.unsqueeze(
            -1
        )

        for k in TOP_K_VALUES:

            correct = (
                top_predictions[:, :, :k]
                == labels_expanded
            ).any(dim=-1).sum().item()

            top_k_correct[k] += correct

        top_k_total += labels.numel()


print()

for k in TOP_K_VALUES:

    accuracy = (
        top_k_correct[k]
        / top_k_total
    )

    print(
        f"Top-{k} Accuracy     : "
        f"{accuracy * 100:.4f}%"
    )

print("=" * 75)

Calculating Top-K Accuracy

Top-1 Accuracy     : 66.7969%
Top-5 Accuracy     : 89.5833%
Top-10 Accuracy     : 93.8802%


In [35]:
# ============================================================
# Cell 17 - Sample Predictions
# ============================================================

print("=" * 75)
print("Sample Validation Predictions")
print("=" * 75)

model.eval()

sample_input, sample_labels = next(
    iter(validation_loader)
)

sample_input = sample_input[:1].to(
    DEVICE
)

sample_labels = sample_labels[:1].to(
    DEVICE
)

with torch.no_grad():

    output = model(
        input_ids=sample_input,
        labels=sample_labels,
    )

    if isinstance(output, tuple):

        logits = output[0]

    else:

        logits = output.logits


predicted_ids = logits.argmax(
    dim=-1
)[0]

actual_ids = sample_labels[0]

print()
print("First 30 token predictions:")
print()

for i in range(
    min(30, predicted_ids.shape[0])
):

    predicted = int(
        predicted_ids[i].item()
    )

    actual = int(
        actual_ids[i].item()
    )

    status = (
        "✓"
        if predicted == actual
        else "✗"
    )

    print(
        f"{i:>3} | "
        f"Predicted: {predicted:>6} | "
        f"Actual: {actual:>6} | "
        f"{status}"
    )

print("=" * 75)

Sample Validation Predictions

First 30 token predictions:

  0 | Predicted:    513 | Actual:    779 | ✗
  1 | Predicted:    239 | Actual:    239 | ✓
  2 | Predicted:    513 | Actual:    513 | ✓
  3 | Predicted:    444 | Actual:    345 | ✗
  4 | Predicted:   5124 | Actual:   5124 | ✓
  5 | Predicted:     17 | Actual:     17 | ✓
  6 | Predicted:    403 | Actual:    403 | ✓
  7 | Predicted:    520 | Actual:    520 | ✓
  8 | Predicted:    235 | Actual:    235 | ✓
  9 | Predicted:    442 | Actual:    442 | ✓
 10 | Predicted:    305 | Actual:    251 | ✗
 11 | Predicted:    221 | Actual:    221 | ✓
 12 | Predicted:    968 | Actual:    968 | ✓
 13 | Predicted:    305 | Actual:    305 | ✓
 14 | Predicted:    448 | Actual:    448 | ✓
 15 | Predicted:    507 | Actual:    507 | ✓
 16 | Predicted:     17 | Actual:     17 | ✓
 17 | Predicted:    740 | Actual:    740 | ✓
 18 | Predicted:    445 | Actual:    445 | ✓
 19 | Predicted:     15 | Actual:     15 | ✓
 20 | Predicted:    378 | Actual:    378

In [36]:
# ============================================================
# Cell 18 - Final Evaluation Summary
# ============================================================

print()
print("=" * 75)
print("MyGPT2 Validation Evaluation Summary")
print("=" * 75)

print(
    f"Checkpoint Step       : "
    f"{checkpoint.get('global_step'):,}"
)

print(
    f"Model Parameters      : "
    f"{total_parameters:,}"
)

print(
    f"Evaluation Documents  : "
    f"{documents_processed:,}"
)

print(
    f"Evaluation Sequences  : "
    f"{len(validation_dataset_torch):,}"
)

print(
    f"Tokens Evaluated      : "
    f"{total_predictions:,}"
)

print()

print(
    f"Training Loss         : "
    f"{training_loss:.6f}"
    if training_loss is not None
    else "Training Loss         : N/A"
)

print(
    f"Validation Loss       : "
    f"{validation_loss:.6f}"
)

print(
    f"Training Perplexity   : "
    f"{training_perplexity:.4f}"
    if training_perplexity is not None
    else "Training Perplexity   : N/A"
)

print(
    f"Validation Perplexity : "
    f"{validation_perplexity:.4f}"
)

print(
    f"Top-1 Accuracy        : "
    f"{top_k_correct[1] / top_k_total * 100:.4f}%"
)

print(
    f"Top-5 Accuracy        : "
    f"{top_k_correct[5] / top_k_total * 100:.4f}%"
)

print(
    f"Top-10 Accuracy       : "
    f"{top_k_correct[10] / top_k_total * 100:.4f}%"
)

print()

if validation_loss < training_loss:

    print(
        "Generalization       : 🟢 VALIDATION LOSS "
        "BELOW TRAINING LOSS"
    )

elif validation_loss < training_loss * 1.10:

    print(
        "Generalization       : 🟢 GOOD"
    )

elif validation_loss < training_loss * 1.30:

    print(
        "Generalization       : 🟡 MODERATE GAP"
    )

else:

    print(
        "Generalization       : 🔴 LARGE TRAIN/VAL GAP"
    )

print()

print(
    "Validation Evaluation : ✅ COMPLETED"
)

print("=" * 75)


MyGPT2 Validation Evaluation Summary
Checkpoint Step       : 10,000
Model Parameters      : 110,025,216
Evaluation Documents  : 569
Evaluation Sequences  : 9
Tokens Evaluated      : 4,608

Training Loss         : 1.389895
Validation Loss       : 1.327773
Training Perplexity   : 4.0144
Validation Perplexity : 3.7726
Top-1 Accuracy        : 66.7969%
Top-5 Accuracy        : 89.5833%
Top-10 Accuracy       : 93.8802%

Generalization       : 🟢 VALIDATION LOSS BELOW TRAINING LOSS

Validation Evaluation : ✅ COMPLETED


In [37]:
# ============================================================
# Cell 19 - Save Evaluation Results
# ============================================================

evaluation_dir = (
    PROJECT_ROOT
    / "artifacts"
    / "evaluation"
)

evaluation_dir.mkdir(
    parents=True,
    exist_ok=True,
)

results = {
    "checkpoint": {
        "path": str(CHECKPOINT_PATH),
        "version": checkpoint.get(
            "checkpoint_version"
        ),
        "global_step": checkpoint.get(
            "global_step"
        ),
    },

    "model": {
        "parameters": total_parameters,
        "vocab_size": VOCAB_SIZE,
        "sequence_length": SEQUENCE_LENGTH,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "intermediate_size": INTERMEDIATE_SIZE,
    },

    "evaluation": {
        "dataset": VALIDATION_DATASET_NAME,
        "documents": documents_processed,
        "sequences": len(
            validation_dataset_torch
        ),
        "tokens": total_predictions,
        "batches": total_batches,
    },

    "metrics": {
        "training_loss": training_loss,
        "validation_loss": validation_loss,
        "training_perplexity": (
            training_perplexity
        ),
        "validation_perplexity": (
            validation_perplexity
        ),
        "top_1_accuracy": (
            top_k_correct[1]
            / top_k_total
        ),
        "top_5_accuracy": (
            top_k_correct[5]
            / top_k_total
        ),
        "top_10_accuracy": (
            top_k_correct[10]
            / top_k_total
        ),
    },

    "runtime": {
        "device": str(DEVICE),
        "validation_time_seconds": (
            validation_time
        ),
    },
}

RESULTS_PATH = (
    evaluation_dir
    / "validation_results_step_10000.json"
)

with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        results,
        file,
        indent=4,
    )

print("=" * 75)
print("Evaluation Results Saved")
print("=" * 75)

print(
    "Results:",
    RESULTS_PATH
)

print()
print("Save                  : ✅ PASSED")
print("=" * 75)

Evaluation Results Saved
Results: D:\Gpt2_v01\artifacts\evaluation\validation_results_step_10000.json

Save                  : ✅ PASSED
